# StarDist for annotating

### Mask generation and Data Preparation

Prepare `dataset/train/images` and `dataset/train/generated_masks` from COCO 1.0 annotations.

In [ ]:
!pip install numpy pillow matplotlib stardist csbdeep tifffile shapely scikit-image imgaug

In [ ]:
from pathlib import Path
import gc
import math

import numpy as np
import tifffile
from csbdeep.utils import normalize

from basic_stardist_model import BasicStarDistModel
from yolo_stardist_model import YoloStarDistModel
from prepare_dataset import prepare_train_dataset, show_image_and_mask_i, show_image_and_mask
from results_to_coco import CocoExporter
from train import (
    DEFAULT_AUGMENT_PARAMS,
    load_image_raw,
    load_train_pairs,
    make_stardist_augmenter,
    normalize_train_images,
    print_runtime_info,
    split_train_val,
)

dataset_name = 'points_test_1'
ORIGINAL_ROOT = Path('dataset/' + dataset_name)
TRAIN_IMAGES_DIR = Path('dataset/train/' + dataset_name + '/images')
TRAIN_MASKS_DIR = Path('dataset/train/' + dataset_name+ '/generated_masks')
TODO_ROOT = Path('dataset/' + "todo1")
ALLOWED_LABELS = {'SterjenArm'}
MODEL_BASEDIR = 'models_' + dataset_name
MIN_TRAIN_SIZE = (128, 128)
REGENERATE_MASKS = True
RETRAIN_MODEL = True

In [ ]:
prepare_train_dataset(
    original_root=ORIGINAL_ROOT,
    train_images_dir=TRAIN_IMAGES_DIR,
    train_masks_dir=TRAIN_MASKS_DIR,
    allowed_labels=ALLOWED_LABELS,
    min_size=MIN_TRAIN_SIZE,
    regenerate_masks=REGENERATE_MASKS,
)

In [ ]:
show_image_and_mask_i(TRAIN_IMAGES_DIR, TRAIN_MASKS_DIR, 0)

### STARDIST training


In [ ]:
print_runtime_info()

In [ ]:
validation_size=6

X_all, Y_all, names_all = load_train_pairs(TRAIN_IMAGES_DIR, TRAIN_MASKS_DIR, min_train_size=MIN_TRAIN_SIZE)
X_all, n_channel, axis_norm = normalize_train_images(X_all)
X_trn, Y_trn, X_val, Y_val = split_train_val(X_all, Y_all, names_all, n_val=validation_size)

print(f'Total pairs: {len(X_all)}')
print(f'Train pairs: {len(X_trn)}')
print(f'Val pairs: {len(X_val)}')
print(f'n_channel_in: {n_channel}')

AUGMENT_PARAMS = dict(DEFAULT_AUGMENT_PARAMS)

In [ ]:
MODEL_WEIGHTS_FILE = 'weights_best.h5'

model = BasicStarDistModel(
    n_channel_in=n_channel,
    model_basedir=MODEL_BASEDIR,
    model_weights_file=MODEL_WEIGHTS_FILE,
    n_rays=12,
    grid=(2, 2),
    train_patch_size=(128, 128),
    train_batch_size=4,
    train_steps_per_epoch=100,
    train_epochs=40,
)
weights_path = model.logdir / MODEL_WEIGHTS_FILE
augmenter = make_stardist_augmenter(**AUGMENT_PARAMS)
if augmenter is None:
    print('Augmentation: disabled')
else:
    print(f"Augmentation: enabled with params={AUGMENT_PARAMS}")

In [ ]:
%%time
if weights_path.exists() and not RETRAIN_MODEL:
    print(f'Reusing existing model weights: {weights_path}')
    model.load_weights(MODEL_WEIGHTS_FILE)
else:
    print('Training started...')
    history = model.train(X_trn, Y_trn, validation_data=(X_val, Y_val), augmenter=augmenter)
    print('Training finished.')

In [ ]:
%%time
if weights_path.exists() and not RETRAIN_MODEL:
    pass
else:
    print('Optimizing thresholds...')
    model.optimize_thresholds(X_val, Y_val)
    print('Thresholds optimized and saved.')

### Prediction + Export


In [ ]:
TODO_ROOT = Path('dataset/todo')
PREDICTED_MASKS_ROOT = Path('dataset/predicted_masks')
PREDICTED_ANNOTATIONS_ROOT = Path('dataset/predicted_annotations')
COCO_CATEGORY_NAME = 'SterjenArm'
COCO_CATEGORY_ID = 1

# Geometry-based simplification tolerance as a fraction of polygon bbox diagonal.
POLYGON_SIMPLIFY_TOLERANCE_REL = 0.07
MAX_POLYGON_POINTS = 200
REGENERATE_RESULT_MASKS = True

# Memory/quality controls for prediction/export
PREDICT_MAX_TILE_SIZE = 1024  # lower -> more tiles -> less peak memory
MIN_INSTANCE_AREA_FOR_EXPORT = 8  # skip tiny artifacts in COCO export


In [ ]:
image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}

todo_image_candidates = sorted(
    p for p in TODO_ROOT.glob('*/images/**/*')
    if p.is_file() and p.suffix.lower() in image_extensions
)
if not todo_image_candidates:
    raise FileNotFoundError('No image files found in dataset/todo/*/images/**')

todo_image_path = todo_image_candidates[0]

relative_to_todo = todo_image_path.relative_to(TODO_ROOT)
todo_folder_name = relative_to_todo.parts[0]

pred_dir = Path('dataset/predicted_masks') / todo_folder_name
pred_dir.mkdir(parents=True, exist_ok=True)

pred_mask_path = pred_dir / f'{todo_image_path.stem}.tiff'

if pred_mask_path.exists() and not REGENERATE_RESULT_MASKS:
    print(f'Reusing existing preview mask: {pred_mask_path}')
else:
    todo_raw = load_image_raw(todo_image_path)
    todo_norm = normalize(todo_raw, 1, 99.8, axis=axis_norm)
    h, w = todo_norm.shape[:2]
    tile_y = max(1, int(math.ceil(h / PREDICT_MAX_TILE_SIZE)))
    tile_x = max(1, int(math.ceil(w / PREDICT_MAX_TILE_SIZE)))
    if todo_norm.ndim == 2:
        n_tiles = (tile_y, tile_x)
    elif todo_norm.ndim == 3:
        n_tiles = (tile_y, tile_x, 1)  # never tile channels
    else:
        raise ValueError(f'Unsupported image ndim for preview prediction: {todo_norm.ndim}')
    labels, details = model.predict_instances(todo_norm, n_tiles=n_tiles)
    tifffile.imwrite(str(pred_mask_path), labels.astype(np.uint16))
    print(f'Generated preview mask: {pred_mask_path} (n_tiles={n_tiles})')
    del todo_raw, todo_norm, labels, details
    gc.collect()

print(f'Preview image: {todo_image_path}')
print(f'Preview mask: {pred_mask_path}')
show_image_and_mask(todo_image_path, pred_mask_path)

In [ ]:
folders = sorted([p for p in TODO_ROOT.iterdir() if p.is_dir()])
coco_exporter = CocoExporter()

for folder_dir in folders:
    coco_exporter.predict_one_folder_to_coco(model, folder_dir,
                                             PREDICTED_MASKS_ROOT,
                                             PREDICTED_ANNOTATIONS_ROOT,
                                             COCO_CATEGORY_NAME,
                                             COCO_CATEGORY_ID,
                                             POLYGON_SIMPLIFY_TOLERANCE_REL,
                                             MAX_POLYGON_POINTS,
                                             PREDICT_MAX_TILE_SIZE,
                                             MIN_INSTANCE_AREA_FOR_EXPORT,
                                             axis_norm,
                                             REGENERATE_RESULT_MASKS=True)